![NVIDIA Logo](images/nvidia.png)

# Capstone Exercise - Build a Multi-Input/Multi-Output (MIMO) Pipeline

## Scenario: Detect Nefarious Activity from Multiple Data Streams

In this section, you will complete an exercise that brings together ideas taught throughout the course materials.

To complete this exercise, you will need to complete a non-linear CyberAI pipeline that is processing two simultaneous data streams.

- Stream 1: Represents telemetry data describing the usage statistics of an NVIDIA GPU. These data can provide a leading indicator of a highjacked system for nefarious use: crypo-mining in this case.
- Stream 2: Represents a stream of network packets which can similarly provide an indication of system compromise.

To detect nefarious usage from these data, two machine learning models have been trained and deployed to Triton Inference Server:

- `abp-nvsmi-xgb`: Trained on GPU telemetry to detect hijacking for cryptomining.
- `abp-pcap-xgb`: Trained on these processed package data to detect nefarous use.

Your job is to complete the Morpheus pipeline below to route these data streams to the corrosponding processing pipelines. The NVSMI data stream must be routed through the NVSMI data processing branch. The PCAP data must be routed through the PCAP data processing branch.

You have been provided the scaffolding but will need to update sections tagged with "FIX-ME", run the pipeline, and generate outputs successfully in order to complete this exercise and receive credit for this course.

Good luck!

---

## Imports

In [ ]:
import logging
import os
import time

from IPython.display import Image
from assessment_helper import grade_assessment

from morpheus.cli.utils import str_to_file_type
from morpheus.config import Config
from morpheus.config import CppConfig
from morpheus.config import PipelineModes

from morpheus.pipeline import Stage, Pipeline
from morpheus.pipeline import Pipeline
from morpheus.pipeline.execution_mode_mixins import GpuAndCpuMixin
from morpheus.pipeline.pass_thru_type_mixin import PassThruTypeMixin
from morpheus.pipeline.single_port_stage import SinglePortStage
from morpheus.pipeline.stage import Stage

from morpheus.stages.general.router_stage import RouterStage
from morpheus.stages.general.monitor_stage import MonitorStage
from morpheus.stages.inference.triton_inference_stage import TritonInferenceStage
from morpheus.stages.input.file_source_stage import FileSourceStage
from morpheus.stages.output.write_to_file_stage import WriteToFileStage
from morpheus.stages.postprocess.add_classifications_stage import AddClassificationsStage
from morpheus.stages.postprocess.serialize_stage import SerializeStage
from morpheus.stages.preprocess.deserialize_stage import DeserializeStage

from morpheus.messages import ControlMessage

from morpheus.utils.logger import configure_logging, reset_logging

import mrc
import mrc.core.operators as ops

import sys
sys.path.append('.')

from preprocess_nvsmi_stage import PreprocessNvsmiStage
from preprocess_pcap_stage import PreprocessPcapStage

---

## Create a Custom Stage that will Add a Metadata Tag to Facilitate Message Routing

Tag each message with metadata that will be used to route the data to the appropriate processing branch (e.g. `nvsmi` or `pcap`).

---

In [ ]:
# @register_stage("route-meta")
class RouteMetadataTaggerStage(PassThruTypeMixin, GpuAndCpuMixin, SinglePortStage):
    
    def __init__(self,
                 config: Config,
                 branch_key: str,
                ):
        super().__init__(config)
        self._branch_key = branch_key
        
    @property
    def name(self) -> str:
        return "route-metadata"

    def accepted_types(self) -> tuple:
        return (ControlMessage, )

    def supports_cpp_node(self) -> bool:
        return False

    # Tag the `ControlMessage` metadata with the branch that should process this message.
    def on_data(self, message: ControlMessage) -> ControlMessage:
        # 이 부분이 수정되었습니다.
        # RouterStage의 key_fn이 "route" 키를 사용하므로, 여기에 branch_key를 설정합니다.
        message.set_metadata("route", self._branch_key)
        return message

    def _build_single(self, builder: mrc.Builder, input_node: mrc.SegmentObject) -> mrc.SegmentObject:
        node = builder.make_node(self.unique_name, ops.map(self.on_data))
        builder.make_edge(input_node, node)

        return node

## Define the Router's Key Function

This function will be used by the router stage to determine which processing path to send the data. Our `router_key_fn` function will consider the tagged metadata to inform its routing decision.

In [ ]:
def router_key_fn(msg: ControlMessage):
    return msg.get_metadata("route")

In [ ]:
reset_logging()
configure_logging(log_level=logging.INFO)

---

## Define the Configuration Object

In [ ]:
# Enable the default logger.
CppConfig.set_should_use_cpp(True)

# Its necessary to get the global config object and configure it for FIL mode.
config = Config()
config.mode = PipelineModes.FIL

# Below properties are specified by the command line.
config.num_threads = os.cpu_count()
config.pipeline_batch_size = 1000000
config.model_max_batch_size = 32768

config.feature_length = 18
config.class_labels = ["probs"]
config.edge_buffer_size = 128

In [ ]:
# Note, because the global configuration freezes during pipeline construction
# We provide another configuration for our PCAP processing branch.

config_pcap = Config()
config_pcap.mode = PipelineModes.FIL

config_pcap.num_threads = os.cpu_count()
config_pcap.pipeline_batch_size = 1000000
config_pcap.model_max_batch_size = 100000
config_pcap.feature_length = 13
config_pcap.class_labels = ["probs"]
config_pcap.edge_buffer_size = 128

---

## Define the Morpheus Pipeline

In [ ]:
# Define a pipeline using `Pipeline` since we will be dealing with nonlinearity
pipeline = Pipeline(config)

# Note, these must be strings
branch_keys=["route_a","route_b"]

# NVSMI INPUT BRANCH
nvsmi_source = pipeline.add_stage(FileSourceStage(config, filename="data/nvsmi.jsonlines", iterative=False))
nvsmi_deserialize = pipeline.add_stage(DeserializeStage(config))
nvsmi_metadata_tagger = pipeline.add_stage(RouteMetadataTaggerStage(config, branch_key="route_a"))


# ABP PCAP INPUT BRANCH
pcap_source = pipeline.add_stage(FileSourceStage(config, filename='data/abp_pcap_dump.jsonlines', iterative=True, file_type=str_to_file_type('auto'), filter_null=False, repeat=10))
pcap_deserialize = pipeline.add_stage(DeserializeStage(config))
pcap_metadata_tagger = pipeline.add_stage(RouteMetadataTaggerStage(config, branch_key="route_b"))

# Define Router Stage
router = pipeline.add_stage(RouterStage(config, keys=branch_keys, key_fn=router_key_fn))


# NVSMI PROCESSING BRANCH
nvsmi_preprocess = pipeline.add_stage(PreprocessNvsmiStage(config, column_file="data/columns_fil.txt"))
nvsmi_inf = pipeline.add_stage(
    TritonInferenceStage(
        config,
        model_name='abp-nvsmi-xgb',
        server_url='triton:8001',
        force_convert_inputs=True,
        use_shared_memory=True
    ))
nvsmi_add_class = pipeline.add_stage(AddClassificationsStage(config, threshold=0.5, labels=["probs"]))
nvsmi_serialize = pipeline.add_stage(SerializeStage(config, include=[], exclude=['^ID$', '^_ts_']))
nvsmi_sink = pipeline.add_stage(WriteToFileStage(config, filename='data/output/nvsmi_out.jsonlines', overwrite=True))

# PCAP PROCESSING BRANCH
pcap_preprocess = pipeline.add_stage(PreprocessPcapStage(config_pcap))
pcap_inf = pipeline.add_stage(
    TritonInferenceStage(
        config_pcap,
        model_name='abp-pcap-xgb',
        server_url='triton:8001',
        force_convert_inputs=True,
        use_shared_memory=True
    ))
pcap_add_class = pipeline.add_stage(AddClassificationsStage(config_pcap, threshold=0.5, labels=["probs"]))
pcap_serialize = pipeline.add_stage(SerializeStage(config_pcap, include=[], exclude=['^ID$', '^_ts_']))
pcap_sink = pipeline.add_stage(WriteToFileStage(config_pcap, filename='data/output/pcap_out.jsonlines', overwrite=True))


# Unified Pipeline Monitor (Interleaved Combine)
pipeline_monitor = pipeline.add_stage(MonitorStage(config, description="Pipeline throughput"))

# Connect Edges to Define the Pipeline Graph
# NVSMI INPUT BRANCH
pipeline.add_edge(nvsmi_source, nvsmi_deserialize)
pipeline.add_edge(nvsmi_deserialize, nvsmi_metadata_tagger)

# PCAP INPUT BRANCH
pipeline.add_edge(pcap_source, pcap_deserialize)
pipeline.add_edge(pcap_deserialize, pcap_metadata_tagger)

# Router In
pipeline.add_edge(nvsmi_metadata_tagger, router)
pipeline.add_edge(pcap_metadata_tagger, router)

# Router Out
pipeline.add_edge(router.output_ports[0], nvsmi_preprocess)
pipeline.add_edge(router.output_ports[1], pcap_preprocess)


# NVSMI PROCESSING BRANCH
pipeline.add_edge(nvsmi_preprocess, nvsmi_inf)
pipeline.add_edge(nvsmi_inf, nvsmi_add_class)
pipeline.add_edge(nvsmi_add_class, nvsmi_serialize)
pipeline.add_edge(nvsmi_serialize, nvsmi_sink)

# PCAP PROCESSING BRANCH
pipeline.add_edge(pcap_preprocess, pcap_inf)
pipeline.add_edge(pcap_inf, pcap_add_class)
pipeline.add_edge(pcap_add_class, pcap_serialize)
pipeline.add_edge(pcap_serialize, pcap_sink)

# Interleave outputs in a unified monitoring stage
pipeline.add_edge(nvsmi_sink, pipeline_monitor)
pipeline.add_edge(pcap_sink, pipeline_monitor)

pipeline.build()

NameError: name 'RouteMetadataTaggerStage' is not defined

## Run the Pipeline

In [ ]:
viz_file = 'pipeline_visualizations/capstone_pipeline.png'
pipeline.visualize(viz_file)

In [ ]:
Image(filename=viz_file)

In [ ]:
await pipeline.run_async()

---

## Grade Your Work

Assuming you were able to successfully build and run the pipeline, execute the following cell to grade your work.

In [ ]:
grade_assessment()

---

## Get Certificate for the Workshop

Having successfully completed the workshop and its assessment, we'd like to offer you a certificate of competency for the workshop.

In your web browser, return to the page where you launched this interactive environment and click the check-mark `ASSESS TASK` button (see the screenshot below). After a few seconds you will get a congratulatory message, after which you can visit your [personal DLI learning page](https://learn.nvidia.com/my-learning) and view your certificate.

![assess](images/assess.png)

---

## Your Valuable Feedback

As a final request, please complete [our workshop survey](https://learn.nvidia.com/courses/course?course_id=course-v1:DLI+C-DS-03+V2&unit=block-v1:DLI+C-DS-03+V2+type@vertical+block@12aa747ec5e241fabf77d4a893ab315b), which should take you no more than a minute or two to complete. Your feedback **greatly** supports our ability to create training content that is high value to developers like you.